# One Postulate in SymPy

This notebook is a computational companion to `paper/one-postulate.tex`.
It explores the same algebraic story as the paper, but it does not replace the
Lean development: the Lean files remain the proof authority, while this
notebook keeps the symbolic calculations easy to inspect.

The paper's claim is that the relativity principle leads to a one-parameter
family of kinematics indexed by `kappa`, and that the Killing form of the
associated homogeneous kinematics algebra sorts that family into three
branches. In the paper's framing, experiment is needed to calibrate the
numerical value of the invariant speed, not to supply its existence as a
separate foundational postulate.

This notebook keeps to the parts of that story that can be reproduced as exact
symbolic calculations in SymPy:

- it starts from the paper's already-derived `1+1` inertial-frame transformation law
- it checks the Galilean limit `kappa -> 0` and a matching `1+1` invariant quadratic form
- it adopts the standard homogeneous kinematics Lie algebra used later in the paper
- it derives the Killing form, the `kappa = 0` degeneracy, and the `kappa > 0` metric check symbolically

Scope boundary:

- what the notebook shows is limited to direct symbolic consequences of the chosen formulas
- the paper's stronger physical readings of the three branches are noted inline, but remain interpretation unless explicitly derived here
- this notebook does not re-derive the full one-parameter family from the relativity principle, and it does not prove the representation-theoretic uniqueness claims invoked in the paper


In [1]:
import sympy as sp

sp.init_printing()

def simplify_matrix(M):
    return M.applyfunc(lambda expr: sp.simplify(sp.together(expr)))

def print_heading(title):
    print("\n" + title)
    print("-" * len(title))


## The postulate and notebook conventions

The paper begins with Einstein's relativity principle and then appeals to the
standard result that, once homogeneity, isotropy, and group composition are
imposed, inertial-frame transformations fall into a one-parameter family.
This notebook enters after that derivation. Its job is not to rebuild the full
argument from first principles, but to make the paper's algebraic claims
inspectable line by line.

All symbols are real unless otherwise stated. Separate positive and negative
versions of `kappa` are introduced only where the sign matters, because the
later cells are designed to let the three sign regimes emerge from the algebra
rather than be assumed in advance.


In [2]:
t, x, v, kappa = sp.symbols('t x v kappa', real=True)
kappa_pos = sp.symbols('kappa_pos', positive=True, real=True)
kappa_neg = sp.symbols('kappa_neg', negative=True, real=True)
c = sp.symbols('c', positive=True, real=True)

gamma = sp.simplify(1 / sp.sqrt(1 - kappa * v**2))
gamma_pos = sp.simplify(1 / sp.sqrt(1 - kappa_pos * v**2))


## What the postulate determines

In the paper, the first concrete output of the postulate is not yet a verdict
about Lorentzian versus Galilean physics, but a family of transformation laws
parametrized by `kappa`. The notebook begins from that already-derived family.
The immediate question is then: what can be read off directly from the
transformation law itself before bringing in the homogeneous kinematics algebra
and the Killing form?


### The derived `1+1` transformation law

The paper emphasizes that the term `kappa v x` in the time transformation is
where the physics first becomes visible: `kappa` controls how strongly space is
mixed into time. The singular set of `gamma` and the `kappa -> 0` limit already
hint at the three branches discussed later, but here the notebook keeps to the
direct symbolic content of the formulas.

The `1+1` transformation law taken from the paper is

\[
t' = \gamma (t - \kappa v x), \qquad
x' = \gamma (x - v t), \qquad
\gamma = \frac{1}{\sqrt{1-\kappa v^2}}.
\]


In [3]:
t_prime = sp.simplify(gamma * (t - kappa * v * x))
x_prime = sp.simplify(gamma * (x - v * t))

Lambda_1p1 = sp.Matrix([
    [gamma, -gamma * kappa * v],
    [-gamma * v, gamma],
])

assert sp.simplify(Lambda_1p1.det() - 1) == 0

print_heading('1+1 transformation law')
print('gamma =')
sp.pprint(gamma)
print("t_prime =")
sp.pprint(t_prime)
print("x_prime =")
sp.pprint(x_prime)
print('det(Lambda_1p1) =')
sp.pprint(sp.simplify(Lambda_1p1.det()))



1+1 transformation law
----------------------
gamma =
       1       
───────────────
   ____________
  ╱      2     
╲╱  - κ⋅v  + 1 
t_prime =
  -κ⋅v⋅x + t   
───────────────
   ____________
  ╱      2     
╲╱  - κ⋅v  + 1 
x_prime =
   -t⋅v + x    
───────────────
   ____________
  ╱      2     
╲╱  - κ⋅v  + 1 
det(Lambda_1p1) =
1


### Newtonian recovery when `kappa -> 0`

One immediate checkpoint from the paper is that `kappa = 0` should recover the
Galilean transformation law. The notebook proves that claim directly with
symbolic limits. This is still a statement about the transformation family
itself; the later claim that the `kappa = 0` algebra loses its internal scale
will only appear once the Killing form is computed.


In [4]:
gamma_at_zero = sp.simplify(sp.limit(gamma, kappa, 0))
t_prime_at_zero = sp.simplify(sp.limit(t_prime, kappa, 0))
x_prime_at_zero = sp.simplify(sp.limit(x_prime, kappa, 0))

assert gamma_at_zero == 1
assert t_prime_at_zero == t
assert x_prime_at_zero == x - v * t

singular_speeds = tuple(sorted(sp.solve(sp.Eq(1 - kappa_pos * v**2, 0), v), key=sp.default_sort_key))
expected_speeds = (-1 / sp.sqrt(kappa_pos), 1 / sp.sqrt(kappa_pos))
assert singular_speeds == expected_speeds

print_heading('Galilean limit')
print('lim_{kappa -> 0} gamma =')
sp.pprint(gamma_at_zero)
print('lim_{kappa -> 0} t_prime =')
sp.pprint(t_prime_at_zero)
print('lim_{kappa -> 0} x_prime =')
sp.pprint(x_prime_at_zero)

print_heading('Finite scale when kappa > 0')
print('gamma has real singular speeds at:')
sp.pprint(singular_speeds)



Galilean limit
--------------
lim_{kappa -> 0} gamma =
1
lim_{kappa -> 0} t_prime =
t
lim_{kappa -> 0} x_prime =
-t⋅v + x

Finite scale when kappa > 0
---------------------------
gamma has real singular speeds at:
⎛  -1         1    ⎞
⎜────────, ────────⎟
⎜  ______    ______⎟
⎝╲╱ κₚₒₛ   ╲╱ κₚₒₛ ⎠


### Warm-up: a `1+1` invariant quadratic form

Before moving to the six-generator homogeneous kinematics algebra, it is useful
to see that the same parameter already supports an invariant quadratic form in
`1+1` dimensions. This is not yet the full spacetime verdict of the paper, but
it previews the kind of invariant structure that the later Lie-algebraic
computation will recover in a more systematic way.


In [5]:
g_1p1 = sp.diag(1, -kappa)
warmup_residual = simplify_matrix(Lambda_1p1.T * g_1p1 * Lambda_1p1 - g_1p1)

assert warmup_residual == sp.zeros(2)

print_heading('1+1 invariant quadratic form')
print('g_(1+1) =')
sp.pprint(g_1p1)
print('Lambda^T g Lambda - g =')
sp.pprint(warmup_residual)



1+1 invariant quadratic form
----------------------------
g_(1+1) =
⎡1  0 ⎤
⎢     ⎥
⎣0  -κ⎦
Lambda^T g Lambda - g =
⎡0  0⎤
⎢    ⎥
⎣0  0⎦


## Can the rules examine themselves?

The paper's decisive move is to stop asking only how coordinates transform and
to ask whether the symmetry rules can diagnose their own structure. To do that,
it passes from the `1+1` transformation law to the homogeneous kinematics Lie
algebra with three rotations `J_i` and three boosts `K_i`. The Killing form is
the algebra's internal self-test: built from the adjoint action, it records
which generators remain visible and which ones collapse.


### Adopted homogeneous kinematics algebra

The notebook now adopts the standard homogeneous kinematics algebra consistent
with the paper's normalization and later formulas. This algebra is not derived
here from the `1+1` transformation law; the notebook takes it as the standard
realization used by the paper once the argument shifts from frame
transformations to the structure of the symmetry generators themselves.

Basis order:

\[
(J_1, J_2, J_3, K_1, K_2, K_3).
\]

Commutators:

\[
[J_i, J_j] = \epsilon_{ijk} J_k, \qquad
[J_i, K_j] = \epsilon_{ijk} K_k, \qquad
[K_i, K_j] = -\kappa \epsilon_{ijk} J_k.
\]


In [6]:
basis = ('J1', 'J2', 'J3', 'K1', 'K2', 'K3')
index = {name: i for i, name in enumerate(basis)}

def levi_civita(i, j, k):
    values = (i, j, k)
    if len(set(values)) < 3:
        return sp.Integer(0)
    inversions = 0
    for a in range(3):
        for b in range(a + 1, 3):
            if values[a] > values[b]:
                inversions += 1
    return sp.Integer(-1 if inversions % 2 else 1)

C = sp.MutableDenseNDimArray.zeros(6, 6, 6)

def basis_bracket(a, b):
    result = [sp.Integer(0)] * 6
    if a == b:
        return sp.Matrix(result)
    if a < 3 and b < 3:
        for k in range(3):
            eps = levi_civita(a + 1, b + 1, k + 1)
            if eps != 0:
                result[k] += eps
    elif a < 3 and b >= 3:
        j = b - 3
        for k in range(3):
            eps = levi_civita(a + 1, j + 1, k + 1)
            if eps != 0:
                result[3 + k] += eps
    elif a >= 3 and b < 3:
        return -basis_bracket(b, a)
    else:
        i = a - 3
        j = b - 3
        for k in range(3):
            eps = levi_civita(i + 1, j + 1, k + 1)
            if eps != 0:
                result[k] += -kappa * eps
    return sp.Matrix(result)

for a in range(6):
    for b in range(6):
        vec = basis_bracket(a, b)
        for c in range(6):
            C[c, a, b] = sp.simplify(vec[c])

def bracket_vector(a, b):
    return sp.Matrix([sp.simplify(C[c, a, b]) for c in range(6)])

assert bracket_vector(index['J1'], index['J2']) == sp.Matrix([0, 0, 1, 0, 0, 0])
assert bracket_vector(index['J1'], index['K2']) == sp.Matrix([0, 0, 0, 0, 0, 1])
assert bracket_vector(index['K1'], index['K2']) == sp.Matrix([0, 0, -kappa, 0, 0, 0])

sample_brackets = {
    '[J1, J2]': bracket_vector(index['J1'], index['J2']),
    '[J1, K2]': bracket_vector(index['J1'], index['K2']),
    '[K1, K2]': bracket_vector(index['K1'], index['K2']),
}

print_heading('Sample commutators in the chosen basis')
for name, vec in sample_brackets.items():
    print(name)
    sp.pprint(vec.T)



Sample commutators in the chosen basis
--------------------------------------
[J1, J2]
[0  0  1  0  0  0]
[J1, K2]
[0  0  0  0  0  1]
[K1, K2]
[0  0  -κ  0  0  0]


### Adjoint action of the generators

To compute the Killing form from first principles, the notebook turns the
commutator table into adjoint matrices. This is the point where the algebra
starts acting on itself, which is exactly the mechanism behind the paper's idea
that the symmetry rules can examine their own structure.


In [7]:
adjoint_matrices = [
    sp.Matrix(6, 6, lambda row, col, a=a: sp.simplify(C[row, a, col]))
    for a in range(6)
]

assert all(M.shape == (6, 6) for M in adjoint_matrices)

print_heading('Example adjoint matrices')
print('ad_J1 =')
sp.pprint(adjoint_matrices[index['J1']])
print('ad_K1 =')
sp.pprint(adjoint_matrices[index['K1']])



Example adjoint matrices
------------------------
ad_J1 =
⎡0  0  0   0  0  0 ⎤
⎢                  ⎥
⎢0  0  -1  0  0  0 ⎥
⎢                  ⎥
⎢0  1  0   0  0  0 ⎥
⎢                  ⎥
⎢0  0  0   0  0  0 ⎥
⎢                  ⎥
⎢0  0  0   0  0  -1⎥
⎢                  ⎥
⎣0  0  0   0  1  0 ⎦
ad_K1 =
⎡0  0  0   0  0   0⎤
⎢                  ⎥
⎢0  0  0   0  0   κ⎥
⎢                  ⎥
⎢0  0  0   0  -κ  0⎥
⎢                  ⎥
⎢0  0  0   0  0   0⎥
⎢                  ⎥
⎢0  0  -1  0  0   0⎥
⎢                  ⎥
⎣0  1  0   0  0   0⎦


### The Killing form as the algebra's self-test

The Killing form is computed from the adjoint representation via

\[
B(X, Y) = \mathrm{tr}(\mathrm{ad}_X \mathrm{ad}_Y).
\]

The diagonal target form is **not** hardcoded as the computation itself; it is
used only as the final symbolic check. Once this matrix is in hand, the sign of
`kappa` becomes an algebraic classification problem rather than an external
physical choice.


In [8]:
killing_matrix = sp.Matrix([
    [sp.simplify((adjoint_matrices[a] * adjoint_matrices[b]).trace()) for b in range(6)]
    for a in range(6)
])
killing_matrix = simplify_matrix(killing_matrix)

expected_killing_matrix = sp.diag(-4, -4, -4, 4 * kappa, 4 * kappa, 4 * kappa)

assert killing_matrix == expected_killing_matrix

print_heading('Derived Killing form')
sp.pprint(killing_matrix)



Derived Killing form
--------------------
⎡-4  0   0    0    0    0 ⎤
⎢                         ⎥
⎢0   -4  0    0    0    0 ⎥
⎢                         ⎥
⎢0   0   -4   0    0    0 ⎥
⎢                         ⎥
⎢0   0   0   4⋅κ   0    0 ⎥
⎢                         ⎥
⎢0   0   0    0   4⋅κ   0 ⎥
⎢                         ⎥
⎣0   0   0    0    0   4⋅κ⎦


## Three verdicts

At this point the notebook has the full Killing matrix. The paper reads that
object as an internal diagnostic that sorts the one-parameter family into three
regimes. The next cells stay on the symbolic side of that argument: determinant,
eigenvalues, block structure, and explicit degeneracy. The language of
Euclidean, Galilean, and Lorentzian branches is introduced only as the paper's
interpretation of those computed facts.


### Algebraic split by the sign of `kappa`

The symbolic matrix already isolates the three sign regimes. What is proved in this
section is purely algebraic: determinant, eigenvalues, block structure, and explicit
degeneracy at `kappa = 0`.


In [9]:
rotation_block = killing_matrix[:3, :3]
boost_block = killing_matrix[3:, 3:]

det_killing = sp.factor(killing_matrix.det())
det_boost = sp.factor(boost_block.det())
eigvals = {sp.simplify(ev): mult for ev, mult in killing_matrix.eigenvals().items()}

killing_at_zero = killing_matrix.subs(kappa, 0)
boost_at_zero = boost_block.subs(kappa, 0)

assert rotation_block == sp.diag(-4, -4, -4)
assert boost_block == sp.diag(4 * kappa, 4 * kappa, 4 * kappa)
assert det_killing == -4096 * kappa**3
assert det_boost == 64 * kappa**3
assert killing_at_zero.det() == 0
assert killing_at_zero.rank() == 3
assert boost_at_zero.rank() == 0
assert sp.ask(sp.Q.negative((4 * kappa_neg))) is True
assert sp.simplify((4 * kappa).subs(kappa, 0)) == 0
assert sp.ask(sp.Q.positive((4 * kappa_pos))) is True

print_heading('Determinant and eigenvalues')
print('det(B) =')
sp.pprint(det_killing)
print('eigenvalues(B) =')
sp.pprint(eigvals)
print('rank(B) at kappa = 0 =', killing_at_zero.rank())



Determinant and eigenvalues
---------------------------
det(B) =
       3
-4096⋅κ 
eigenvalues(B) =
{-4: 3, 4⋅κ: 3}
rank(B) at kappa = 0 = 3


### `kappa < 0`: the negative branch

The notebook shows two direct facts about this branch: the boost block
`4 kappa I_3` is negative definite when `kappa < 0`, and the singular-speed
equation has no real solutions there. The paper reads those algebraic facts as
the Euclidean or compact branch, with no lightcones and no causal ordering.
That geometric reading is not derived here, but this is the point in the
computation where the paper's negative-`kappa` verdict attaches.


### `kappa = 0`: the algebra goes blind to boosts

The paper's sharpest internal diagnostic is at `kappa = 0`: the rotation block
survives, but the boost block vanishes. The notebook proves exactly that
collapse. The paper then interprets it as the loss of an internally determined
velocity-space ruler and the return of background structures such as absolute
time. Those broader claims are marked as interpretation whenever they appear.


In [10]:
assert boost_at_zero == sp.zeros(3)
assert rotation_block.subs(kappa, 0).rank() == 3

print_heading('Rotation block')
sp.pprint(rotation_block)
print_heading('Boost block')
sp.pprint(boost_block)
print_heading('Boost block at kappa = 0')
sp.pprint(boost_at_zero)



Rotation block
--------------
⎡-4  0   0 ⎤
⎢          ⎥
⎢0   -4  0 ⎥
⎢          ⎥
⎣0   0   -4⎦

Boost block
-----------
⎡4⋅κ   0    0 ⎤
⎢             ⎥
⎢ 0   4⋅κ   0 ⎥
⎢             ⎥
⎣ 0    0   4⋅κ⎦

Boost block at kappa = 0
------------------------
⎡0  0  0⎤
⎢       ⎥
⎢0  0  0⎥
⎢       ⎥
⎣0  0  0⎦


### Reading the zero branch in the paper

Taken together, the exact Galilean limit `t' = t` from above and the vanished
boost block computed here are the notebook's two direct symbolic signatures of
the `kappa = 0` regime. The paper reads this combination as Galilean structure:
time stays untouched by boosts, the boost sector carries no intrinsic scale,
and additional background structure has to be supplied from outside. That last
step is the paper's interpretation, not a symbolic theorem proved in this
notebook.


### `kappa > 0`: the boost sector fixes a scale

For positive `kappa`, the same boost block becomes `4 kappa delta_ij`, and the
transformation factor `gamma` develops real singular speeds at
`v = +/- 1/sqrt(kappa)`. The notebook proves those statements directly. The
paper reads `1/sqrt(kappa)` as the finite invariant-speed scale selected by the
algebra itself.

The identification of that scale with a physically meaningful universal speed is
therefore partly symbolic and partly interpretive: the notebook shows the scale
appearing, while the paper explains what role it plays.


In [11]:
delta3 = sp.eye(3)
assert boost_block == 4 * kappa * delta3

speed_scale = sp.simplify(1 / sp.sqrt(kappa_pos))

print_heading('Boost block proportionality')
sp.pprint(boost_block)
print('=')
sp.pprint(4 * kappa * delta3)

print_heading('Scale extracted for kappa > 0')
print('1/sqrt(kappa_pos) =')
sp.pprint(speed_scale)
print('real singular speeds of gamma =')
sp.pprint(singular_speeds)



Boost block proportionality
---------------------------
⎡4⋅κ   0    0 ⎤
⎢             ⎥
⎢ 0   4⋅κ   0 ⎥
⎢             ⎥
⎣ 0    0   4⋅κ⎦
=
⎡4⋅κ   0    0 ⎤
⎢             ⎥
⎢ 0   4⋅κ   0 ⎥
⎢             ⎥
⎣ 0    0   4⋅κ⎦

Scale extracted for kappa > 0
-----------------------------
1/sqrt(kappa_pos) =
   1    
────────
  ______
╲╱ κₚₒₛ 
real singular speeds of gamma =
⎛  -1         1    ⎞
⎜────────, ────────⎟
⎜  ______    ______⎟
⎝╲╱ κₚₒₛ   ╲╱ κₚₒₛ ⎠


### `kappa > 0`: spacetime geometry closes the loop

The paper's positive-branch claim is not only that velocity space acquires a
finite scale, but that spacetime itself carries an invariant quadratic form.
Here the notebook checks the proposed metric under an explicit x-axis boost.
The paper's stronger claims about uniqueness, irreducibility, and lightcones are
not derived here; they are the higher-level interpretation attached to this
symbolic invariant.

The invariant metric proposed in the paper is

\[
g \propto \mathrm{diag}(1, -\kappa, -\kappa, -\kappa),
\]

equivalently

\[
ds^2 = \kappa^{-1} dt^2 - dx^2 - dy^2 - dz^2.
\]


In [12]:
g = sp.diag(1, -kappa, -kappa, -kappa)
g_rescaled = sp.diag(1 / kappa, -1, -1, -1)
assert simplify_matrix(g - kappa * g_rescaled) == sp.zeros(4)

Lambda_x = sp.Matrix([
    [gamma, -gamma * kappa * v, 0, 0],
    [-gamma * v, gamma, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

metric_residual = simplify_matrix(Lambda_x.T * g * Lambda_x - g)
assert metric_residual == sp.zeros(4)

print_heading('Invariant metric')
sp.pprint(g)
print('Lambda_x^T g Lambda_x - g =')
sp.pprint(metric_residual)



Invariant metric
----------------
⎡1  0   0   0 ⎤
⎢             ⎥
⎢0  -κ  0   0 ⎥
⎢             ⎥
⎢0  0   -κ  0 ⎥
⎢             ⎥
⎣0  0   0   -κ⎦
Lambda_x^T g Lambda_x - g =
⎡0  0  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎣0  0  0  0⎦


## Structure and scale

By this point the notebook has isolated the paper's structural claim. The
algebraic side yields the branch split, the `kappa = 0` collapse, the positive
speed scale `1/sqrt(kappa)`, and an explicit invariant metric candidate. What
remains is not structural existence but numerical calibration: once the
physically relevant invariant speed is identified with `c`, the free parameter
is fixed by `kappa = 1/c^2`.


### Calibration against the measured speed `c`

This is the paper's final distinction between structure and measurement. The
notebook does not need experiment to make the symbolic scale appear; experiment
enters only when one assigns that scale the measured value `c`. Symbolically,
that is the substitution `kappa = 1/c^2`.


In [13]:
kappa_calibrated = sp.simplify(1 / c**2)
calibrated_speed = sp.simplify(1 / sp.sqrt(kappa_calibrated))

assert sp.simplify(calibrated_speed - c) == 0

print_heading('Calibration')
print('kappa =')
sp.pprint(kappa_calibrated)
print('1/sqrt(kappa) =')
sp.pprint(calibrated_speed)



Calibration
-----------
kappa =
0.0400000000000000
1/sqrt(kappa) =
5.00000000000000


## Lean crosswalk

The notebook checks the same objects that Lean formalizes. Read this table as a
map from computational checkpoint to proof surface.

| Notebook checkpoint | Lean anchor |
| --- | --- |
| Transformation law and `kappa -> 0` limit | `OnePostulate/SpacetimeMatrices.lean` |
| Bracket family | `matrix_bracket_JJ`, `matrix_bracket_JK`, `matrix_bracket_KK`, `kinematic_bracket_table` |
| Killing form computation | `OnePostulate/KillingForm.lean`, `killing_form_diag`, `boost_killing_form_eq` |
| Boost and spacetime consequences | `killing_restricts_to_metric`, `spacetime_metric_invariant`, `phase1_selection_summary` |

The notebook stays subordinate to those Lean files: it helps readers explore
the formulas, but it does not supply the proof.


## What the notebook proves and what the paper concludes

The paper ends by distinguishing structural determination from empirical
calibration. The final section preserves that distinction in notebook form:
first as a separation between direct symbolic checks, Lean formalization, and
paper-level interpretation, then as a compact regime table aligning the three.


In [ ]:
proved_claims = [
    "The 1+1 transformation law has the exact Galilean limit t_prime = t and x_prime = x - v*t as kappa -> 0.",
    "For kappa > 0, gamma has real singular speeds at v = +/- 1/sqrt(kappa).",
    "The adopted homogeneous kinematics algebra yields the Killing form diag(-4 I3, 4 kappa I3).",
    "The boost block is exactly 4 kappa I3.",
    "At kappa = 0 the boost block vanishes and the Killing form drops rank from 6 to 3.",
    "The metric diag(1, -kappa, -kappa, -kappa) is invariant under the explicit x-axis boost.",
    "The substitution kappa = 1/c^2 calibrates the already-derived scale so that 1/sqrt(kappa) = c.",
]

lean_claims = [
    "matrix_bracket_JJ / matrix_bracket_JK / matrix_bracket_KK",
    "kinematic_bracket_table",
    "killing_form_diag",
    "boost_killing_form_eq",
    "killing_restricts_to_metric",
    "spacetime_metric_invariant",
    "phase1_selection_summary",
]

interpreted_claims = [
    "That kappa < 0 should be read as a compact Euclidean branch with no causal structure.",
    "That kappa = 0 is physically unacceptable because it requires background structure.",
    "That kappa > 0 is the uniquely admissible physical branch.",
    "That the invariant metric is unique by Schur's lemma.",
    "That the full one-parameter family follows from the relativity postulate alone.",
]

print_heading('Symbolically checked here')
for item in proved_claims:
    print('- '+ item)

print_heading('Formalized in Lean')
for item in lean_claims:
    print('- '+ item)

print_heading('Higher-level interpretation from the paper')
for item in interpreted_claims:
    print('- '+ item)

print("\nNote: the notebook adopts the standard homogeneous kinematics algebra consistent with the paper; it does not derive that algebra from the transformation law alone.")


## Regime comparison

The notebook's symbolic checkpoints line up with the three branches of the paper as follows.

| Regime | Direct symbolic consequences | Lean anchor | Paper interpretation |
| --- | --- | --- | --- |
| `kappa < 0` | boost block = `4*kappa*I3 < 0`; Killing form nondegenerate for `kappa != 0`; `gamma` has no real singular speed | `negative_kappa_selects_euclidean`; `negative_kappa_no_nonzero_null_vectors` | Euclidean branch; no lightcone or causal ordering |
| `kappa = 0` | boost block = `0`; `gamma = 1`; Galilean limit holds; Killing form rank drops to `3` | `zero_kappa_selects_galilean`; `reducible_of_kappa_zero` | Galilean branch; invariant `dt^2` only; background structure remains |
| `kappa > 0` | boost block = `4*kappa*I3 > 0`; `gamma` singular at `v = +/- 1/sqrt(kappa)`; metric `diag(1,-kappa,-kappa,-kappa)` is boost-invariant | `positive_kappa_selects_lorentz`; `positive_kappa_gives_finite_real_invariant_speed`; `spacetime_metric_congruent_stdLorentz_of_kappa_pos` | Lorentz branch; finite invariant speed; lightcones and spacetime unification |

The Lean anchor is the proof authority. The notebook only provides a computational lens on the same branch structure.